___

# TEXT MINING - #6 TOPIC MODELLING
___

# BERTopic: Modern Topic Modeling with Transformers

BERTopic is a topic modeling technique that leverages transformer-based embeddings and clustering algorithms to create dense topic representations.

## Why BERTopic?

Traditional methods (LDA, NMF) have limitations:
- **Bag of Words**: Ignore word order and semantics
- **Short texts**: Struggle with tweets, titles, short reviews
- **Fixed vocabulary**: Can't handle out-of-vocabulary words

BERTopic addresses these by using:
1. **Sentence Transformers**: Capture semantic meaning
2. **UMAP**: Dimensionality reduction preserving local structure
3. **HDBSCAN**: Density-based clustering (no need to specify K!)
4. **c-TF-IDF**: Class-based TF-IDF for topic representation

## Setup

In [ ]:
# Install required packages
# !pip install bertopic
# !pip install sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Optional: for comparison
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print("All packages loaded successfully!")

## Load Data

We'll use the same NPR dataset for comparison with LDA/NMF.

In [ ]:
# Load the NPR dataset
url = 'https://raw.githubusercontent.com/nluninja/text-mining-dataviz/refs/heads/main/4.%20Topic%20Modeling/npr.csv'
npr = pd.read_csv(url)
documents = npr['Article'].tolist()

print(f"Loaded {len(documents)} documents")
print(f"\nExample document (first 500 chars):")
print(documents[0][:500])

---
## How BERTopic Works

BERTopic follows a 4-step pipeline:

```
Documents → [1. Embeddings] → [2. UMAP] → [3. HDBSCAN] → [4. c-TF-IDF] → Topics
```

### Step 1: Document Embeddings
Convert documents to dense vector representations using Sentence Transformers.

### Step 2: Dimensionality Reduction (UMAP)
Reduce embedding dimensions while preserving local structure.

### Step 3: Clustering (HDBSCAN)
Find clusters of similar documents. Documents not fitting any cluster become "outliers" (topic -1).

### Step 4: Topic Representation (c-TF-IDF)
Extract representative words for each cluster using class-based TF-IDF.

---
## 1. Basic BERTopic Usage

In [ ]:
# Create BERTopic model with default settings
# Note: First run will download the embedding model (~400MB)
topic_model = BERTopic(verbose=True)

# Fit the model
# This may take a few minutes depending on your hardware
topics, probs = topic_model.fit_transform(documents)

print(f"\nNumber of topics found: {len(set(topics)) - 1}")  # -1 to exclude outlier topic
print(f"Number of outliers (topic -1): {topics.count(-1)}")

In [ ]:
# View topic information
topic_info = topic_model.get_topic_info()
print("Top 10 Topics:")
topic_info.head(10)

In [ ]:
# Get words for a specific topic
print("Topic 0 - Top words:")
topic_model.get_topic(0)

In [ ]:
# Display all topics
print("All Topics with their top words:\n")
for topic_id in range(min(10, len(topic_model.get_topics()) - 1)):  # Show first 10 topics
    words = [word for word, _ in topic_model.get_topic(topic_id)[:8]]
    print(f"Topic {topic_id}: {', '.join(words)}")

---
## 2. Visualizations

BERTopic provides several built-in visualizations.

In [ ]:
# Visualize topics as bar charts
fig = topic_model.visualize_barchart(top_n_topics=8)
fig.show()

In [ ]:
# Visualize topic hierarchy
fig = topic_model.visualize_hierarchy()
fig.show()

In [ ]:
# Visualize topic similarity (heatmap)
fig = topic_model.visualize_heatmap()
fig.show()

In [ ]:
# Visualize documents in 2D space
# This shows how documents cluster together
fig = topic_model.visualize_documents(documents[:2000])  # Subset for performance
fig.show()

In [ ]:
# Visualize term importance per topic
fig = topic_model.visualize_topics()
fig.show()

---
## 3. Customizing BERTopic

You can customize each component of the pipeline.

### 3.1 Custom Embedding Model

In [ ]:
# Use a specific sentence transformer model
# Options: "all-MiniLM-L6-v2" (fast), "all-mpnet-base-v2" (better quality)

from sentence_transformers import SentenceTransformer

# Fast model for experimentation
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Pre-compute embeddings (useful if you want to experiment with different settings)
embeddings = embedding_model.encode(documents, show_progress_bar=True)

In [ ]:
# Create model with pre-computed embeddings
topic_model_custom = BERTopic(embedding_model=embedding_model, verbose=True)
topics, probs = topic_model_custom.fit_transform(documents, embeddings)

print(f"Number of topics: {len(set(topics)) - 1}")

### 3.2 Custom UMAP Parameters

In [ ]:
from umap import UMAP

# Custom UMAP settings
umap_model = UMAP(
    n_neighbors=15,      # Balance between local and global structure
    n_components=5,      # Reduce to 5 dimensions
    min_dist=0.0,        # Allow tight clusters
    metric='cosine',     # Use cosine similarity
    random_state=42
)

topic_model_umap = BERTopic(
    umap_model=umap_model,
    embedding_model=embedding_model,
    verbose=True
)
topics, probs = topic_model_umap.fit_transform(documents, embeddings)

### 3.3 Custom HDBSCAN Parameters

In [ ]:
from hdbscan import HDBSCAN

# Custom HDBSCAN settings
hdbscan_model = HDBSCAN(
    min_cluster_size=50,      # Minimum documents per topic
    min_samples=10,           # Core point threshold
    metric='euclidean',
    cluster_selection_method='eom',  # 'eom' or 'leaf'
    prediction_data=True
)

topic_model_hdbscan = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    embedding_model=embedding_model,
    verbose=True
)
topics, probs = topic_model_hdbscan.fit_transform(documents, embeddings)

print(f"Number of topics: {len(set(topics)) - 1}")
print(f"Outliers: {topics.count(-1)} ({100*topics.count(-1)/len(topics):.1f}%)")

### 3.4 Controlling Number of Topics

In [ ]:
# Option 1: Set nr_topics to reduce topics after clustering
topic_model_reduced = BERTopic(
    nr_topics=10,  # Reduce to 10 topics
    embedding_model=embedding_model,
    verbose=True
)
topics, probs = topic_model_reduced.fit_transform(documents, embeddings)

print(f"Number of topics: {len(set(topics)) - 1}")

In [ ]:
# Option 2: Use "auto" to automatically reduce similar topics
topic_model_auto = BERTopic(
    nr_topics="auto",
    embedding_model=embedding_model,
    verbose=True
)
topics, probs = topic_model_auto.fit_transform(documents, embeddings)

print(f"Number of topics: {len(set(topics)) - 1}")

### 3.5 Custom Vectorizer for Topic Representation

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Custom vectorizer with n-grams
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),      # Include bigrams
    stop_words='english',
    min_df=5,
    max_df=0.9
)

topic_model_ngram = BERTopic(
    vectorizer_model=vectorizer_model,
    embedding_model=embedding_model,
    verbose=True
)
topics, probs = topic_model_ngram.fit_transform(documents, embeddings)

# Check topics - now may include bigrams
print("Topic 0 with bigrams:")
topic_model_ngram.get_topic(0)

---
## 4. Working with Topics

### 4.1 Get Representative Documents

In [ ]:
# Get representative documents for each topic
representative_docs = topic_model.get_representative_docs()

# Show representative docs for topic 0
print("Representative documents for Topic 0:\n")
for i, doc in enumerate(representative_docs[0][:3]):
    print(f"Doc {i+1}: {doc[:300]}...\n")

### 4.2 Find Topics for New Documents

In [ ]:
# Transform new documents (inference)
new_documents = [
    "The president announced new economic policies today.",
    "Scientists discovered a new species of deep sea fish.",
    "The basketball team won the championship game last night."
]

new_topics, new_probs = topic_model.transform(new_documents)

for doc, topic in zip(new_documents, new_topics):
    topic_words = [w for w, _ in topic_model.get_topic(topic)[:5]]
    print(f"Document: {doc[:50]}...")
    print(f"  → Topic {topic}: {', '.join(topic_words)}\n")

### 4.3 Find Similar Topics

In [ ]:
# Find topics similar to a given topic
similar_topics, similarity = topic_model.find_topics("economy", top_n=5)

print("Topics most related to 'economy':")
for topic, sim in zip(similar_topics, similarity):
    words = [w for w, _ in topic_model.get_topic(topic)[:5]]
    print(f"  Topic {topic} (similarity: {sim:.3f}): {', '.join(words)}")

### 4.4 Reduce Outliers

In [ ]:
# Outliers (topic -1) can be reduced by assigning them to nearest topics
print(f"Outliers before: {topics.count(-1)}")

# Reduce outliers using c-TF-IDF strategy
new_topics = topic_model.reduce_outliers(documents, topics, strategy="c-tf-idf")
print(f"Outliers after (c-tf-idf): {list(new_topics).count(-1)}")

# Alternative: use embeddings strategy
new_topics_emb = topic_model.reduce_outliers(documents, topics, strategy="embeddings")
print(f"Outliers after (embeddings): {list(new_topics_emb).count(-1)}")

### 4.5 Merge Topics

In [ ]:
# Merge similar topics manually
# First, let's see which topics might be similar
fig = topic_model.visualize_hierarchy()
fig.show()

In [ ]:
# Merge topics (example: merge topics 1 and 2 if they seem similar)
# topic_model.merge_topics(documents, [1, 2])

# Or merge multiple groups
# topics_to_merge = [[1, 2], [3, 4, 5]]  # Merge 1+2 and 3+4+5
# topic_model.merge_topics(documents, topics_to_merge)

---
## 5. Topics Over Time (Dynamic Topic Modeling)

In [ ]:
# If you have timestamps, you can analyze how topics evolve
# Example with synthetic timestamps:

import datetime

# Create synthetic timestamps (in reality, you'd have actual dates)
base_date = datetime.datetime(2020, 1, 1)
timestamps = [base_date + datetime.timedelta(days=i % 365) for i in range(len(documents))]

# Analyze topics over time
topics_over_time = topic_model.topics_over_time(documents, timestamps, nr_bins=12)
topics_over_time.head()

In [ ]:
# Visualize topics over time
fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=5)
fig.show()

---
## 6. Saving and Loading Models

In [ ]:
# Save model
topic_model.save("bertopic_model")

# Load model
loaded_model = BERTopic.load("bertopic_model")

# Use loaded model
print("Loaded model topics:")
loaded_model.get_topic_info().head()

---
## 7. BERTopic vs LDA/NMF Comparison

In [ ]:
# Quick comparison table
comparison = pd.DataFrame({
    'Aspect': [
        'Approach',
        'Semantics',
        'Short texts',
        'Number of topics (K)',
        'Speed',
        'Interpretability',
        'Out-of-vocabulary',
        'GPU acceleration',
        'Outlier handling'
    ],
    'LDA': [
        'Probabilistic (Bayesian)',
        'Bag of Words (no semantics)',
        'Poor (not enough words)',
        'Must specify',
        'Slow for large corpora',
        'Good (probabilities)',
        'Cannot handle',
        'Limited',
        'No explicit handling'
    ],
    'NMF': [
        'Matrix factorization',
        'Bag of Words (no semantics)',
        'Poor',
        'Must specify',
        'Fast',
        'Good (sparse)',
        'Cannot handle',
        'Yes (via sklearn)',
        'No explicit handling'
    ],
    'BERTopic': [
        'Clustering + c-TF-IDF',
        'Semantic embeddings',
        'Excellent',
        'Automatic (via HDBSCAN)',
        'Medium (embedding cost)',
        'Excellent',
        'Handles via embeddings',
        'Yes (sentence-transformers)',
        'Explicit outlier topic (-1)'
    ]
})

print(comparison.to_string(index=False))

---
## 8. Best Practices & Tips

### When to Use BERTopic
- Short documents (tweets, reviews, titles)
- When semantic understanding matters
- When you don't know how many topics exist
- When you need to handle outliers explicitly

### When to Use LDA/NMF Instead
- Very large corpora (millions of documents) - embeddings are expensive
- When you need probabilistic interpretation (LDA)
- Limited computational resources
- When bag-of-words is sufficient

### Tips for Better Results
1. **Pre-compute embeddings** for faster experimentation
2. **Tune min_cluster_size** in HDBSCAN to control topic granularity
3. **Use bigrams** in vectorizer for more descriptive topics
4. **Reduce outliers** if too many documents are unassigned
5. **Merge similar topics** for cleaner results
6. **Use GPU** if available for faster embedding computation

---
## Key Takeaways

1. **BERTopic** combines transformer embeddings with clustering for modern topic modeling

2. **Automatic K**: HDBSCAN finds the number of topics automatically (unlike LDA/NMF)

3. **Semantic understanding**: Captures meaning, not just word co-occurrence

4. **Short text friendly**: Works well with tweets, titles, short reviews

5. **Outlier handling**: Explicitly identifies documents that don't fit any topic

6. **Flexible**: Each component (embeddings, UMAP, HDBSCAN, vectorizer) can be customized

7. **Rich visualizations**: Built-in interactive visualizations

---
## References

- BERTopic documentation: https://maartengr.github.io/BERTopic/
- Paper: Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. arXiv:2203.05794
- Sentence Transformers: https://www.sbert.net/
- UMAP: https://umap-learn.readthedocs.io/
- HDBSCAN: https://hdbscan.readthedocs.io/